## Automatic video annotation using SAM3 video text-prompt model

In [ ]:
import gc
import json
import shutil
import sys
import torch
import numpy as np
from pathlib import Path
from PIL import Image
from skimage import measure
from tqdm import tqdm

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "src":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SAM3_DIR = PROJECT_ROOT / "pkg" / "sam3"
if str(SAM3_DIR) not in sys.path:
    sys.path.insert(0, str(SAM3_DIR))

In [ ]:
from sam3.model_builder import build_sam3_video_predictor

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

gpus_to_use = range(torch.cuda.device_count())
predictor = build_sam3_video_predictor(checkpoint_path="../weights/sam3/sam3.pt",
    gpus_to_use=gpus_to_use)
print("SAM3 video predictor loaded")

### Configure class to text prompt mapping

In [ ]:
# class_id -> (Chinese label, English text prompt for SAM3)
class_config = {
    0: ("person", "person"),
    1: ("car", "car"),
}

# Video path: either a folder of JPEG frames or a .mp4 file
video_path = str(PROJECT_ROOT / "datasets/task_video/frames")
annotation_save_path = PROJECT_ROOT / "datasets/task_video/annotations/annotations.json"
frame_save_dir = PROJECT_ROOT / "datasets/task_video/frame"  # save extracted frames here

prompt_frame_idx = 0  # which frame to apply the text prompt on

# Frame sampling: only annotate every Nth frame to reduce computation
source_fps = 30       # original video FPS (for reference)
target_fps = 5        # desired annotation FPS (set <= source_fps)
frame_stride = max(1, source_fps // target_fps)  # e.g. 30//5=6, annotate every 6th frame

print(f"Video path: {video_path}")
print(f"Frame save dir: {frame_save_dir}")
print(f"Prompt frame index: {prompt_frame_idx}")
print(f"Source FPS={source_fps}, target FPS={target_fps} -> frame_stride={frame_stride}")
print(f"Classes: {[v[0] for v in class_config.values()]}")

## Run SAM3 video text-prompt inference & save COCO annotations

In [ ]:
def propagate_and_collect(predictor, session_id, sampled_set, frame_to_image_id,
                           coco_annotations, class_id, img_w, img_h, ann_id_start, min_area=0):
    """
    Stream propagation results and extract COCO annotations on-the-fly.
    Only keeps sampled frames' outputs; discards others immediately.
    Returns (next_ann_id, count_this_class).
    """
    ann_id = ann_id_start
    obj_count = 0
    for response in predictor.handle_stream_request(
        request=dict(
            type="propagate_in_video",
            session_id=session_id,
        )
    ):
        frame_idx = response["frame_index"]
        if frame_idx not in sampled_set:
            continue  # skip unsampled frames, free their outputs immediately
        outputs = response["outputs"]
        segms = outputs_to_coco_annotations(outputs, img_w, img_h)
        for seg, bbox, area in segms:
            if area < min_area:
                continue
            coco_annotations.append({
                "id": ann_id,
                "image_id": frame_to_image_id[frame_idx],
                "category_id": class_id,
                "segmentation": [seg],
                "bbox": bbox,
                "area": area,
                "iscrowd": 0,
            })
            ann_id += 1
            obj_count += 1
    return ann_id, obj_count


def outputs_to_coco_annotations(outputs, image_w, image_h):
    """
    Convert a single frame's outputs dict to list of (segmentation, bbox_xywh, area).
    outputs keys: out_obj_ids, out_binary_masks, out_boxes_xywh, out_probs
    """
    results = []
    if outputs is None or len(outputs.get("out_obj_ids", [])) == 0:
        return results

    obj_ids = outputs["out_obj_ids"]
    masks = outputs["out_binary_masks"]
    boxes_xywh = outputs["out_boxes_xywh"]

    for i in range(len(obj_ids)):
        mask_np = np.asarray(masks[i])
        if mask_np.ndim == 2:
            mask_np = mask_np
        else:
            mask_np = mask_np.squeeze(0) if mask_np.ndim == 3 else mask_np
        mask_np = mask_np.astype(np.uint8)
        if not mask_np.any():
            continue

        contours = measure.find_contours(mask_np.astype(np.float64), 0.5)
        if not contours:
            continue
        contour = max(contours, key=len)
        contour_xy = np.fliplr(contour)
        segmentation = contour_xy.flatten().tolist()

        # boxes_xywh: [x, y, w, h] in pixel coords
        x, y, w, h = boxes_xywh[i]
        bbox = [float(x), float(y), float(w), float(h)]

        poly = contour_xy
        area = 0.5 * abs(np.dot(poly[:, 0], np.roll(poly[:, 1], 1)) -
                         np.dot(poly[:, 1], np.roll(poly[:, 0], 1)))

        results.append((segmentation, bbox, float(area)))

    return results


def get_video_frame_paths(video_path):
    """Return sorted list of frame file paths, or integer range for mp4."""
    import glob
    if video_path.endswith(".mp4"):
        import cv2
        cap = cv2.VideoCapture(video_path)
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        cap.release()
        return list(range(frame_count))
    else:
        frame_paths = glob.glob(str(Path(video_path) / "*.jpg"))
        try:
            frame_paths.sort(key=lambda p: int(Path(p).stem))
        except ValueError:
            frame_paths.sort()
        return [Path(p) for p in frame_paths]


def get_frame_size(video_path):
    """Get (width, height) of the first frame."""
    import glob
    import cv2
    if video_path.endswith(".mp4"):
        cap = cv2.VideoCapture(video_path)
        ret, frame = cap.read()
        cap.release()
        if ret:
            return frame.shape[1], frame.shape[0]
    else:
        frame_paths = glob.glob(str(Path(video_path) / "*.jpg"))
        if frame_paths:
            img = Image.open(frame_paths[0])
            return img.size
    return 1920, 1080


def save_frame_image(video_path, frame_idx, frame_save_dir):
    """Save a single frame to frame_save_dir as frame_{idx:05d}.jpg."""
    import cv2
    frame_save_dir = Path(frame_save_dir)
    frame_save_dir.mkdir(parents=True, exist_ok=True)
    out_name = f"frame_{frame_idx:05d}.jpg"
    out_path = frame_save_dir / out_name
    if out_path.exists():
        return out_name  # already saved
    if video_path.endswith(".mp4"):
        cap = cv2.VideoCapture(video_path)
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()
        cap.release()
        if ret:
            cv2.imwrite(str(out_path), frame)
    else:
        import glob
        frame_paths = sorted(glob.glob(str(Path(video_path) / "*.jpg")))
        if frame_idx < len(frame_paths):
            shutil.copy2(frame_paths[frame_idx], out_path)
    return out_name


def cleanup_gpu():
    """Release GPU memory and run garbage collection."""
    gc.collect()
    torch.cuda.empty_cache()


def make_coco_video_annotation(video_path, class_config, predictor,
                                prompt_frame_idx, save_path,
                                frame_stride=1, frame_save_dir=None, min_area=0):
    """
    Run SAM3 video text-prompt on a video (per-class sessions),
    collect per-frame masks on-the-fly, and save COCO JSON.
    """
    frame_paths = get_video_frame_paths(video_path)
    num_frames = len(frame_paths)
    img_w, img_h = get_frame_size(video_path)

    sampled_frame_idxs = list(range(0, num_frames, frame_stride))
    sampled_set = set(sampled_frame_idxs)

    print(f"Video: {num_frames} total frames, {img_w}x{img_h}")
    print(f"Frame stride={frame_stride} -> annotating {len(sampled_frame_idxs)} frames")

    # Build COCO images list + save frames
    coco_images = []
    frame_to_image_id = {}
    for img_id, frame_idx in enumerate(tqdm(sampled_frame_idxs, desc="Saving frames"), start=1):
        file_name = save_frame_image(video_path, frame_idx, frame_save_dir) if frame_save_dir else f"{frame_idx:05d}.jpg"
        frame_to_image_id[frame_idx] = img_id
        coco_images.append({
            "id": img_id,
            "file_name": file_name,
            "frame_idx": frame_idx,
            "width": img_w,
            "height": img_h,
        })

    # Per-class sessions: stream-propagate + extract annotations on-the-fly
    coco_annotations = []
    ann_id = 1
    total_objs = 0

    for class_id, (label, text_prompt) in tqdm(class_config.items(), desc="Classes"):
        print(f"Class [{label}]: text prompt = '{text_prompt}'")

        # Start session
        response = predictor.handle_request(
            request=dict(type="start_session", resource_path=video_path)
        )
        session_id = response["session_id"]

        # Add text prompt
        predictor.handle_request(
            request=dict(
                type="add_prompt",
                session_id=session_id,
                frame_index=prompt_frame_idx,
                text=text_prompt,
            )
        )

        # Propagate + extract annotations on the fly (no full dict buffered)
        ann_id, obj_count = propagate_and_collect(
            predictor, session_id, sampled_set, frame_to_image_id,
            coco_annotations, class_id, img_w, img_h, ann_id, min_area)
        total_objs += obj_count
        print(f"  Found {obj_count} instance-frame annotations")

        # Close session + cleanup GPU
        predictor.handle_request(
            request=dict(type="close_session", session_id=session_id)
        )
        cleanup_gpu()

    # Assemble and save COCO JSON
    coco_output = {
        "images": coco_images,
        "annotations": coco_annotations,
        "categories": [
            {"id": cid, "name": label_, "supercategory": "object"}
            for cid, (label_, _) in class_config.items()
        ],
    }

    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(coco_output, f, indent=2, ensure_ascii=False)

    # Save labels.txt
    labels_path = save_path.parent / "labels.txt"
    with open(labels_path, "w", encoding="utf-8") as f_labels:
        for cid in sorted(class_config.keys()):
            f_labels.write(f"{class_config[cid][0]}
")

    print(f"COCO annotation saved to {save_path}")
    print(f"   images={len(coco_images)}, "
          f"annotations={len(coco_annotations)}, "
          f"categories={len(class_config)}")
    print(f"   total objects found (across sampled frames): {total_objs}")

In [ ]:
# Run auto-annotation
if not annotation_save_path.exists():
    annotation_save_path.parent.mkdir(parents=True, exist_ok=True)
    annotation_save_path.touch()

make_coco_video_annotation(
    video_path=video_path,
    class_config=class_config,
    predictor=predictor,
    prompt_frame_idx=prompt_frame_idx,
    save_path=annotation_save_path,
    frame_stride=frame_stride,
    frame_save_dir=frame_save_dir,
    min_area=0,
)

In [ ]:
# Clean up: shutdown predictor to free GPU resources
predictor.shutdown()